# Job Market Trends - Enhanced ETL Pipeline

This notebook implements a structured ETL pipeline that integrates job postings with company metrics, industries, and **skills**. It includes stage-wise logging and data quality tracking.

In [1]:
import os
import datetime
import pandas as pd
import numpy as np
import kagglehub

class StepLogger:
    def __init__(self):
        self.logs = []

    def log(self, step_name, details, df=None):
        row_count = len(df) if df is not None else "N/A"
        entry = {
            'Timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'Step': step_name,
            'Details': details,
            'Row Count': row_count
        }
        self.logs.append(entry)
        print(f"[{entry['Timestamp']}] {step_name}: {details} (Rows: {row_count})")

    def get_summary(self):
        return pd.DataFrame(self.logs)

logger = StepLogger()

## 1. Data Ingestion

In [2]:
def load_datasets():
    logger.log("Ingestion", "Locating dataset from Kaggle")
    data_path = kagglehub.dataset_download("arshkon/linkedin-job-postings")

    datasets = {
        'jobs': pd.read_csv(os.path.join(data_path, 'postings.csv')),
        'companies': pd.read_csv(os.path.join(data_path, 'companies', 'companies.csv')),
        'employees': pd.read_csv(os.path.join(data_path, 'companies', 'employee_counts.csv')),
        'industries': pd.read_csv(os.path.join(data_path, 'companies', 'company_industries.csv')),
        'ind_labels': pd.read_csv(os.path.join(data_path, 'mappings', 'industries.csv')),
        'job_skills': pd.read_csv(os.path.join(data_path, 'jobs', 'job_skills.csv')),
        'skills': pd.read_csv(os.path.join(data_path, 'mappings', 'skills.csv'))
    }

    for name, df in datasets.items():
        logger.log("Ingestion", f"Loaded {name}", df=df)

    return datasets

data_bundle = load_datasets()

[2026-04-21 14:44:59] Ingestion: Locating dataset from Kaggle (Rows: N/A)
[2026-04-21 14:45:04] Ingestion: Loaded jobs (Rows: 123849)
[2026-04-21 14:45:04] Ingestion: Loaded companies (Rows: 24473)
[2026-04-21 14:45:04] Ingestion: Loaded employees (Rows: 35787)
[2026-04-21 14:45:04] Ingestion: Loaded industries (Rows: 24375)
[2026-04-21 14:45:04] Ingestion: Loaded ind_labels (Rows: 422)
[2026-04-21 14:45:04] Ingestion: Loaded job_skills (Rows: 213768)
[2026-04-21 14:45:04] Ingestion: Loaded skills (Rows: 35)


## 2. Merging Datasets

In [3]:
def merge_data(bundle):
    jobs = bundle['jobs']
    companies = bundle['companies']
    employees = bundle['employees']
    industries = bundle['industries']
    ind_labels = bundle['ind_labels']
    job_skills = bundle['job_skills']
    skills = bundle['skills']

    # 1. Join Jobs with Companies
    df = jobs.merge(companies[['company_id', 'name', 'company_size']], left_on='company_id', right_on='company_id', how='left')
    logger.log("Merge", "Joined jobs with company metadata", df=df)

    # 2. Join with Latest Employee Counts
    latest_employees = employees.sort_values('time_recorded').groupby('company_id').tail(1)
    df = df.merge(latest_employees[['company_id', 'employee_count']], on='company_id', how='left')
    logger.log("Merge", "Joined with latest employee counts", df=df)

    # 3. Join with Industries
    # Convert 'industry' column to numeric for consistent merging
    industries['industry'] = pd.to_numeric(industries['industry'], errors='coerce')
    industry_full = industries.merge(ind_labels, left_on='industry', right_on='industry_id', how='left')
    comp_industries = industry_full.groupby('company_id')['industry_name'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    df = df.merge(comp_industries, on='company_id', how='left')
    logger.log("Merge", "Joined with company industries", df=df)

    # 4. Join with Skills
    skills_full = job_skills.merge(skills, on='skill_abr', how='left')
    job_skills_summary = skills_full.groupby('job_id')['skill_name'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    df = df.merge(job_skills_summary, on='job_id', how='left')
    logger.log("Merge", "Joined with job skills", df=df)

    return df

merged_df = merge_data(data_bundle)

[2026-04-21 14:45:04] Merge: Joined jobs with company metadata (Rows: 123849)
[2026-04-21 14:45:04] Merge: Joined with latest employee counts (Rows: 123849)
[2026-04-21 14:45:05] Merge: Joined with company industries (Rows: 123849)
[2026-04-21 14:45:10] Merge: Joined with job skills (Rows: 123849)


## 3. Cleaning & Validation

In [4]:
def clean_data(df):
    df = df.drop_duplicates()

    Q1 = df['normalized_salary'].quantile(0.05)
    Q3 = df['normalized_salary'].quantile(0.95)
    df = df[df['normalized_salary'].between(Q1, Q3) | df['normalized_salary'].isna()]


    # Type Conversions
    numeric_cols = ['views', 'applies', 'min_salary', 'max_salary', 'normalized_salary', 'employee_count']
    for col in numeric_cols:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')

    time_cols = ['listed_time', 'expiry', 'original_listed_time']
    for col in time_cols:
        if col in df.columns: df[col] = pd.to_datetime(df[col], errors='coerce', unit='ms')

    text_cols = ['title', 'location', 'name', 'industry_name', 'skill_name', 'formatted_work_type', 'formatted_experience_level']
    for col in text_cols:
        if col in df.columns: df[col] = df[col].astype('string').str.strip().replace(['nan', 'None'], pd.NA)

    def sort_industries(val):
        if pd.isna(val): return val
        return ', '.join(sorted(val.split(', ')))

    df['industry_name'] = df['industry_name'].apply(sort_industries)

    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    print("\nNull % per column:")
    print(null_pct[null_pct > 0].sort_values(ascending=False))

    logger.log("Cleaning", "Cleaned types and standardized strings", df=df)
    return df

cleaned_df = clean_data(merged_df)


Null % per column:
closed_time                   99.12
skills_desc                   98.00
med_salary                    95.54
remote_allowed                87.76
applies                       81.21
max_salary                    77.34
min_salary                    77.34
compensation_type             72.88
currency                      72.88
normalized_salary             72.88
pay_period                    72.88
posting_domain                32.22
application_url               29.58
formatted_experience_level    23.77
fips                          22.03
zip_code                      16.75
company_size                   5.37
industry_name                  1.52
skill_name                     1.42
name                           1.39
employee_count                 1.39
company_id                     1.39
company_name                   1.39
views                          1.38
description                    0.01
dtype: float64
[2026-04-21 14:45:10] Cleaning: Cleaned types and standardized st

## 4. Feature Engineering

In [5]:
def transform_data(df):
    df['year'] = df['listed_time'].dt.year
    df['month'] = df['listed_time'].dt.month

    df['remote_allowed'] = df['remote_allowed'].fillna(0)
    df['is_remote'] = df['remote_allowed'].map({1.0: 'Remote', 0.0: 'Onsite'})

    df['views'] = df['views'].fillna(0)
    df['applies'] = df['applies'].fillna(0)
    df['demand_score'] = df['views'] + (df['applies'] * 2)



    def group_exp(x):
        if pd.isna(x): return 'Not Specified'
        x = str(x).lower()
        if 'senior' in x or 'director' in x or 'executive' in x: return 'Senior'
        if 'entry' in x or 'intern' in x: return 'Entry'
        return 'Mid-level'

    df['experience_group'] = df['formatted_experience_level'].apply(group_exp)
    df['final_company_name'] = df['name'].fillna(df['company_name']).fillna('Unknown')

    df['city'] = df['location'].str.split(',').str[0].str.strip()
    df['state'] = df['location'].str.split(',').str[-1].str.strip()
    df['location_remote_flag'] = df['location'].str.contains('Remote|Anywhere', case=False, na=False)

    df['log_salary'] = np.log1p(df['normalized_salary'])

    df['industry_name'] = df['industry_name'].fillna('Unknown')
    df['experience_group'] = df['experience_group'].fillna('Not Specified')
    df['is_remote'] = df['is_remote'].fillna('Unknown')

    logger.log("Transformation", "Generated features and resolved company names", df=df)
    return df

transformed_df = transform_data(cleaned_df)

[2026-04-21 14:45:11] Transformation: Generated features and resolved company names (Rows: 120447)


## 5. Export

In [6]:
def save_data(df):
    cols = [
        'job_id', 'final_company_name', 'title', 'location', 'industry_name',
        'skill_name', 'company_size', 'employee_count', 'is_remote',
        'experience_group', 'normalized_salary', 'demand_score', 'year', 'month'
    ]
    final_df = df[cols].copy()
    final_df.rename(columns={'skill_name': 'top_skills'}, inplace=True)
    final_df.to_csv("../Dataset/linkedin_jobs_dashboard_ready.csv", index=False)
    logger.log("Export", "Saved enriched dataset to CSV with Skills", df=final_df)
    df_skills = df[['job_id', 'final_company_name', 'industry_name',
                     'normalized_salary', 'experience_group', 'is_remote', 'skill_name']].copy()
    df_skills.rename(columns={'skill_name': 'top_skills'}, inplace=True)
    df_skills = df_skills.assign(skill=df_skills['top_skills'].str.split(', ')).explode('skill')
    df_skills['skill'] = df_skills['skill'].str.strip()
    df_skills = df_skills[df_skills['skill'].notna() & (df_skills['skill'] != '')]
    df_skills.drop(columns='top_skills', inplace=True)
    df_skills.to_csv("../Dataset/linkedin_skills_exploded.csv", index=False)
    logger.log("Export", "Saved exploded skills CSV", df=df_skills)

    return final_df


final_df = save_data(transformed_df)

[2026-04-21 14:45:11] Export: Saved enriched dataset to CSV with Skills (Rows: 120447)
[2026-04-21 14:45:11] Export: Saved exploded skills CSV (Rows: 200112)


In [7]:
logger.get_summary()

,Timestamp,Step,Details,Row Count
0,2026-04-21 14:44:59,Ingestion,Locating dataset from Kaggle,N/A
1,2026-04-21 14:45:04,Ingestion,Loaded jobs,123849
2,2026-04-21 14:45:04,Ingestion,Loaded companies,24473
3,2026-04-21 14:45:04,Ingestion,Loaded employees,35787
4,2026-04-21 14:45:04,Ingestion,Loaded industries,24375
5,2026-04-21 14:45:04,Ingestion,Loaded ind_labels,422
6,2026-04-21 14:45:04,Ingestion,Loaded job_skills,213768
7,2026-04-21 14:45:04,Ingestion,Loaded skills,35
8,2026-04-21 14:45:04,Merge,Joined jobs with company metadata,123849
9,2026-04-21 14:45:04,Merge,Joined with latest employee counts,123849
